# Quoridor AlphaZero — Colab training

Trains the Quoridor bot by self-play with checkpoints on Google Drive, so the run
survives Colab disconnects and GPU preemption.

**How to use**

1. `Runtime > Change runtime type > GPU` (T4 is fine).
2. `Runtime > Run all`.
3. After any crash or disconnect: just `Run all` again — training **resumes
   exactly** where it left off (same episode, optimizer state, replay buffer,
   RNG streams) from the checkpoint on Drive.

The game environment runs on the C++ engine (`quoridor/`) exposed through
pybind11 — a lockstep-verified, drop-in replacement for the pure-Python
reference env (see `alphazero/test_cpp_backend.py`).

In [12]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__)
print('GPU:', gpus[0].name if gpus else
      'NONE — enable one via Runtime > Change runtime type > GPU')

TensorFlow 2.20.0
GPU: /physical_device:GPU:0


## 1 — Google Drive

Checkpoints, network weights and training plots are written here, so they
persist across runtimes.

In [13]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2 — Code

In [14]:
import os

REPO = '/content/quoridor-bot'
if not os.path.exists(REPO):
    !git clone https://github.com/sihaowu1/quoridor-bot {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 4.98 KiB | 1.66 MiB/s, done.
From https://github.com/sihaowu1/quoridor-bot
   a82bc8b..9922baa  main       -> origin/main
Updating a82bc8b..9922baa
Fast-forward
 README.md                            |  37 ++--
 alphazero/notebook.ipynb             | 318 ++++++++++++++++++++++++++++++++++-
 alphazero/test_alphazero_quoridor.py |  20 ++-
 quoridor/main.cpp                    |   2 +
 4 files changed, 337 insertions(+), 40 deletions(-)
/content/quoridor-bot


## 3 — Build the C++ engine

`quoridor/build_ext.py` compiles the pybind11 extension to
`alphazero/quoridor_engine.*.so` (on Colab it uses the preinstalled g++; no
zig needed here). The last line confirms the pipeline actually selected the
C++ backend — it must print `alphazero.quoridor_cpp`.

In [15]:
!pip install -q pybind11
!python quoridor/build_ext.py
!python -c "from alphazero.game_config import Quoridor; print('engine backend:', Quoridor.__module__)"

c++ -std=c++17 -O3 -DNDEBUG -fPIC -shared -fvisibility=hidden -I/usr/local/lib/python3.12/dist-packages/pybind11/include -I/usr/include/python3.12 /content/quoridor-bot/quoridor/quoridor.cpp /content/quoridor-bot/quoridor/bindings.cpp -o /content/quoridor-bot/alphazero/quoridor_engine.cpython-312-x86_64-linux-gnu.so
built alphazero/quoridor_engine.cpython-312-x86_64-linux-gnu.so
import smoke test passed
engine backend: alphazero.quoridor_cpp


## 4 — Cross-validate the C++ engine

Replays rule-targeted positions and hundreds of random games through both
backends in lockstep, asserting bit-identical observations, rewards, legal
actions and exceptions at every ply, then prints a speed comparison. Takes a
few minutes; skip it once you trust the build.

In [16]:
!python -m alphazero.test_cpp_backend --bench

_test_constructor_invariants: ok
_test_jump_positions: ok
_test_wall_rule_positions: ok
_test_wins_both_players: ok
_test_illegal_and_exception_parity: ok
_test_truncation_parity: ok
_test_clone_independence: ok
_test_random_lockstep_5x5: ok
_test_random_lockstep_jump_heavy: ok
_test_random_lockstep_9x9: ok
all tests passed
5x5: 100 random games, 4713 plies | python 0.17s, cpp 0.03s (5.6x faster, 159k plies/s)
9x9: 20 random games, 3378 plies | python 0.83s, cpp 0.04s (21.9x faster, 89k plies/s)


## 5 — Configure the run

| env variable          | meaning                                             |
|-----------------------|-----------------------------------------------------|
| `AZ_CHECKPOINT_DIR`   | Drive folder for checkpoints, weights and plots     |
| `AZ_EPISODES`         | total self-play episodes for the run                |
| `AZ_CHECKPOINT_EVERY` | save frequency in episodes (1 = safest)             |
| `AZ_BACKEND`          | `cpp` = require the C++ engine (fail loudly)        |

The board size / wall count live in `alphazero/game_config.py`
(currently the full 9×9 game with 10 walls).

In [17]:
import os

CKPT_DIR = '/content/drive/MyDrive/quoridor-checkpoints'
os.environ['AZ_CHECKPOINT_DIR'] = CKPT_DIR
os.environ['AZ_EPISODES'] = '2000'
os.environ['AZ_CHECKPOINT_EVERY'] = '1'
os.environ['AZ_BACKEND'] = 'cpp'

os.makedirs(CKPT_DIR, exist_ok=True)
state = sorted(f for f in os.listdir(CKPT_DIR) if f.endswith('_train_state.pkl'))
print('checkpoint dir:', CKPT_DIR)
print('will resume from:', state[0] if state else 'nothing yet — fresh run')

checkpoint dir: /content/drive/MyDrive/quoridor-checkpoints
will resume from: nothing yet — fresh run


## 6 — Train

Safe to interrupt and re-run at any time: the driver saves the complete
training state atomically after every episode and auto-resumes from the
latest readable checkpoint (a `.bak` fallback covers a save that was cut
off mid-write).

In [ ]:
!python -m alphazero.run

2026-07-02 21:02:38.108937: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1783026158.110436   16890 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
no checkpoint in /content/drive/MyDrive/quoridor-checkpoints; starting fresh
episode 1: winner +0 in 194 moves, last 1: P1 0.00 / draw 1.00, avg length 194.0
I0000 00:00:1783026314.220150   16890 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
episode 2: winner +0 in 194 moves, last 2: P1 0.00 / draw 1.00, avg length 194.0
episode 3: winner +0 in 194 moves, last 3: P1 0.00 / draw 1.00, avg length 194.0
episode 4: winner +1 in 43 moves, last 4: P1 0.25 / draw 0.75, avg length 156.2
episode

: 

## 7 — Progress plots

Refreshed on Drive at every evaluation, so this cell can be re-run while
training is going (or from a different machine mounting the same Drive).

In [ ]:
import glob
import os

from IPython.display import Image, display

pngs = sorted(glob.glob(os.path.join(os.environ['AZ_CHECKPOINT_DIR'], '*.png')))
if not pngs:
    print('no plots yet — they appear after the first evaluation')
for png in pngs:
    print(os.path.basename(png))
    display(Image(filename=png))